In [42]:
import yfinance as yf
import pandas as pd

# Carregar os tickers do arquivo CSV
tickers_df = pd.read_csv('b3.csv')
tickers = tickers_df['Ticker'].tolist()

# Dicionário para armazenar os dados de cada empresa
data_dict = {
    'Ticker': [],
    'Quantidade de Ações no Mercado': [],
    'Margem Bruta': [],
    'Ativo Circulante': [],
    'Passivo Circulante': [],
    'Divida Líquida': [],
    'Patrimonio Líquido': [],
    'Receita Liquida': [],
    'Receita com Serviços': [],
    'EBITDA': [],
    'Lucro Liquido': [],
    'Razão Corrente': [],
    'Margem EBITDA': [],
    'Margem Líquida': [],
    'ROE': [],
    'ROA': [],
    'Índice de Endividamento': [],
    'Índice de Liquidez Corrente': [],
    'Índice de Liquidez Seca': [],
    'Margem de Contribuição': []
}

# Função para obter os dados financeiros de cada ticker
def get_financial_data(ticker):
    try:
        stock = yf.Ticker(ticker)
        data = stock.info
        return data
    except Exception as e:
        print(f"Erro ao obter dados para {ticker}: {e}")
        return None

# Iterar sobre os tickers e obter os dados financeiros de cada um
for ticker in tickers:
    data = get_financial_data(ticker)
    if data is not None:
        data_dict['Ticker'].append(ticker)
        data_dict['Quantidade de Ações no Mercado'].append(float(data.get('sharesOutstanding', '0')))
        data_dict['Margem Bruta'].append(float(data.get('grossMargins', '0')))
        data_dict['Ativo Circulante'].append(float(data.get('totalAssets', '0')))
        data_dict['Passivo Circulante'].append(float(data.get('totalLiabilities', '0')))
        data_dict['Divida Líquida'].append(float(data.get('netDebt', '0')))
        data_dict['Patrimonio Líquido'].append(float(data.get('totalStockholderEquity', '0')))
        data_dict['Receita Liquida'].append(float(data.get('totalRevenue', '0')))
        data_dict['Receita com Serviços'].append(float(data.get('totalRevenue', '0')))  # Não tenho dados específicos para serviços, usando totalRevenue
        data_dict['EBITDA'].append(float(data.get('ebitda', '0')))
        data_dict['Lucro Liquido'].append(float(data.get('netIncome', '0')))
        data_dict['Razão Corrente'] = [ativo / passivo if passivo != 0 else 0 for ativo, passivo in zip(data_dict['Ativo Circulante'], data_dict['Passivo Circulante'])]
        data_dict['Margem EBITDA'] = [ebitda / receita if receita != 0 else 0 for ebitda, receita in zip(data_dict['EBITDA'], data_dict['Receita Liquida'])]
        data_dict['Margem Líquida'] = [lucro / receita if receita != 0 else 0 for lucro, receita in zip(data_dict['Lucro Liquido'], data_dict['Receita Liquida'])]
        data_dict['ROE'] = [lucro / patrimonio if patrimonio != 0 else 0 for lucro, patrimonio in zip(data_dict['Lucro Liquido'], data_dict['Patrimonio Líquido'])]
        data_dict['ROA'] = [lucro / ativo if ativo != 0 else 0 for lucro, ativo in zip(data_dict['Lucro Liquido'], data_dict['Ativo Circulante'])]
        data_dict['Índice de Endividamento'] = [divida / patrimonio if patrimonio != 0 else 0 for divida, patrimonio in zip(data_dict['Divida Líquida'], data_dict['Patrimonio Líquido'])]
        data_dict['Índice de Liquidez Corrente'] = [ativo / passivo if passivo != 0 else 0 for ativo, passivo in zip(data_dict['Ativo Circulante'], data_dict['Passivo Circulante'])]


# Verificar se todas as listas no dicionário têm o mesmo comprimento
lengths = {key: len(value) for key, value in data_dict.items()}
max_length = max(lengths.values())
for key, length in lengths.items():
    if length < max_length:
        data_dict[key] += ['0'] * (max_length - length)

# Criar DataFrame a partir do dicionário
df = pd.DataFrame(data_dict)

# Exportar os dados para um arquivo Excel
df.to_excel('D:/export_yahoo.xlsx', index=False)
